# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

> 실행 완료 Notebook `chapter04/chapter04.ipynb`에 실습 결과와 해석을 작성했다.

## 0. 제출 정보

- 이름: Jongeun Kim
- GitHub ID: risa0122
- 작성일: 2026-10-07
- 최종 제출 URL: https://github.com/risa0122/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb

## 1. 질문과 필요한 데이터 선택

### 내가 확인하려는 질문

어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?

완료 주문만 대상으로 `quantity × unit_price`를 합산하여 카테고리별로 비교한다. 주문 건수는 `order_id.nunique()`로 계산한다. 금액은 완료 주문 상세 금액이며 실제 회사 매출액·순매출·이익과 구분한다.

### 사용한 파일/컬럼

| 파일 | 주요 컬럼 | 사용 목적 |
|---|---|---|
| order_items.csv | order_item_id, order_id, product_id, quantity, unit_price | 상세 금액 계산 |
| orders.csv | order_id, customer_id, order_date, order_status | 주문 상태·일자·고객 연결 |
| products.csv | product_id, product_name, category | 상품과 카테고리 연결 |
| customers.csv | customer_id | 고객 ID 연결 여부 확인 |

[공식 실습 가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter04/chapter04.md)의 가상 쇼핑몰 데이터를 사용했다. 입력 자료는 Chapter 03에서 사용한 강의 저장소의 `data/raw/` CSV 4개다. 샘플 데이터는 재생성하지 않았으며, 파일 해시로 동일한 자료인지 확인했다. 고객 이름 등 불필요한 개인 속성은 불러오지 않았다.

실행 환경은 Python 3.12.14, pandas 3.0.5, VS Code Notebook이다. 기존 강의 저장소의 `chapter04/chapter04.ipynb`에서 실행한다.

In [1]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", 120)

candidates = [Path.cwd(), *Path.cwd().parents, Path.home() / "llm-data-analysis-course"]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "data/raw/orders.csv").is_file()
     and (p / "requirements.txt").is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("기존 llm-data-analysis-course 폴더에서 실행해 주세요.")

DATA_DIR = PROJECT_ROOT / "data/raw"
EXPECTED_HASHES = {
    "customers": "a663be98960246a436ce7e1ad81284760000ac5ba0d52a7adb3465e6b0457550",
    "products": "46ff06753ade3e71011ee08dd366148dcd9c36521f111840cead0a9d1047e405",
    "orders": "0df461755b933af9f5413427a1c160ca2101b24a5fcd2c4f68d5249a90b9a5e3",
    "order_items": "a7a5c3942b0886c66cb38cc6a10fdfa68dbcde90e8642585dee49c4bd104063a"
}
for name, expected in EXPECTED_HASHES.items():
    path = DATA_DIR / f"{name}.csv"
    assert path.is_file(), f"누락 파일: {name}.csv"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, (
        f"{name}.csv가 해석에 사용한 자료와 다릅니다. 데이터를 확인해 주세요."
    )

# 고객 이름·성별·연령·도시는 이번 질문에 필요하지 않아 읽지 않는다.
customers = pd.read_csv(DATA_DIR / "customers.csv", usecols=["customer_id"])
products = pd.read_csv(DATA_DIR / "products.csv")
orders = pd.read_csv(DATA_DIR / "orders.csv")
order_items = pd.read_csv(DATA_DIR / "order_items.csv")
raw_items = order_items.copy(deep=True)
datasets = dict(customers=customers, products=products, orders=orders, order_items=order_items)
print("Python:", sys.version.split()[0], "/ pandas:", pd.__version__)
print("입력 파일 4개 확인 / 기존 데이터 해시 일치: PASS")

Python: 3.12.14 / pandas: 3.0.5
입력 파일 4개 확인 / 기존 데이터 해시 일치: PASS


In [2]:
required = {
    "customers": ["customer_id"],
    "products": ["product_id", "product_name", "category"],
    "orders": ["order_id", "customer_id", "order_date", "order_status"],
    "order_items": ["order_item_id", "order_id", "product_id", "quantity", "unit_price"],
}
for name, columns in required.items():
    assert set(columns).issubset(datasets[name].columns), f"{name}: 필수 컬럼 누락"
    assert datasets[name][columns].notna().all().all(), f"{name}: 필수 값 결측"
structure = pd.DataFrame([
    {"파일": name, "행": len(df), "사용 컬럼": ", ".join(required[name])}
    for name, df in datasets.items()
])
display(structure)
display(orders["order_status"].value_counts(dropna=False).rename_axis("order_status").to_frame("order_count"))
assert set(orders["order_status"].unique()) == {"completed", "cancelled", "refunded"}
assert not order_items["order_item_id"].duplicated().any()

,파일,행,사용 컬럼
0,customers,150,customer_id
1,products,100,"product_id, product_name, category"
2,orders,300,"order_id, customer_id, order_date, order_status"
3,order_items,764,"order_item_id, order_id, product_id, quantity, unit_price"


,order_count
order_status,
completed,184
cancelled,64
refunded,52


![분석 질문 및 계산 기준](images/step01_question.png)

### 결과 관찰

고객 150명, 상품 100개, 주문 300건, 주문 상세 764행을 확인했다. 주문 상태는 completed 184건, cancelled 64건, refunded 52건이다.

### 나의 해석과 판단

주문 상세에는 주문 상태와 카테고리가 없어 orders와 products를 연결해야 한다. 거래 시점의 금액을 계산하기 위해 상품 마스터의 price 대신 주문 상세의 unit_price를 선택했다. 취소·환불 주문을 제외하고 동일한 completed 범위를 유지하기로 했다.

### 업무·분석적 의미

주문 상태, 금액 계산식, 집계 단위를 먼저 정하면 보고서마다 서로 다른 기준으로 계산하는 문제를 줄일 수 있다.

### 한계와 추가 확인 사항

강의용 가상 데이터이며 실제 고객이나 시장을 대표하지 않는다. 원천 시스템의 주문 상태 정의와 거래 단가 반영 기준은 별도 확인이 필요하다.

## 2. 필터링·정렬·파생 컬럼

- 적용한 필터 조건: orders를 연결한 뒤 `order_status == "completed"`만 선택했다. cancelled·refunded는 제외했다.
- 정렬 기준: 카테고리·상품·고객은 total_sales 내림차순, 월은 시간순으로 정렬했다.
- 만든 파생 컬럼: 상세 금액 line_total, 주문 월 order_month, 카테고리 금액 비중 sales_share_pct.
- `line_total` 계산식: `quantity × unit_price`.

In [3]:
# 주문 상세별 금액 계산
order_items = raw_items.copy()
for col in ["quantity", "unit_price"]:
    assert pd.api.types.is_numeric_dtype(order_items[col]), f"{col}: 숫자형 필요"
    assert np.isfinite(order_items[col]).all(), f"{col}: 무한대 또는 결측"
assert order_items["quantity"].gt(0).all()
assert order_items["quantity"].mod(1).eq(0).all()
assert order_items["unit_price"].ge(0).all()

order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]
display(order_items[["order_id", "product_id", "quantity", "unit_price", "line_total"]].head())
print("주문 상세:", len(order_items), "행 / 계산 누락:", order_items["line_total"].isna().sum())
print("아직 주문 상태를 연결하지 않았으므로 전체 합계를 completed 금액으로 해석하지 않음")

,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


주문 상세: 764 행 / 계산 누락: 0
아직 주문 상태를 연결하지 않았으므로 전체 합계를 completed 금액으로 해석하지 않음


![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰

764개 주문 상세 행에 line_total을 생성했다. quantity와 unit_price는 숫자형이며 계산 결과에 결측이 없었다. 주문 상태 연결 후 completed 상세 474행을 집계에 사용했다.

### 나의 해석과 판단

파생 컬럼 생성 단계에는 취소·환불 주문도 포함되어 있다. 상태 필터를 생략하면 전체 금액이 255,610,000으로 계산되어 completed 금액 148,990,000과 달라진다. 계산식뿐 아니라 포함 범위도 명확해야 한다.

### 업무·분석적 의미

상세 금액을 같은 식으로 계산하고 주문 상태를 일관되게 제한하면 집계 단위를 바꾸어도 비교 가능한 결과를 만들 수 있다.

### 한계와 추가 확인 사항

할인·세금·배송비가 별도 컬럼으로 제공되지 않아 반영 여부를 확인할 수 없다.

## 3. merge 검증

- 병합한 데이터: order_items와 orders, completed 주문 상세와 products.
- 사용한 key: orders 연결은 order_id, products 연결은 product_id.
- `validate` 결과: 두 병합 모두 `many_to_one` 검증을 통과했다. orders.order_id와 products.product_id 중복은 각각 0건이다.
- `indicator` 결과: 두 병합 모두 `indicator=True`로 확인했다. 모두 both이며 미매칭은 0건이다.
- 병합 전/후 행 수: orders 병합은 764 → 764행, products 병합은 completed 상세 474 → 474행이다.

In [4]:
orders_key_duplicates = int(orders["order_id"].duplicated().sum())
products_key_duplicates = int(products["product_id"].duplicated().sum())
assert orders_key_duplicates == 0
assert products_key_duplicates == 0
assert not customers["customer_id"].duplicated().any()

order_sales = order_items.merge(
    orders[["order_id", "customer_id", "order_date", "order_status"]],
    on="order_id", how="left", validate="many_to_one", indicator=True
)
orders_merge_counts = order_sales["_merge"].value_counts()
assert len(order_sales) == len(order_items)
assert order_sales["_merge"].eq("both").all()
assert order_sales["order_item_id"].is_unique
assert set(order_sales["order_item_id"]) == set(order_items["order_item_id"])
order_sales = order_sales.drop(columns="_merge")

order_sales["order_date"] = pd.to_datetime(order_sales["order_date"], errors="coerce")
date_failures = int(order_sales["order_date"].isna().sum())
assert date_failures == 0, "날짜 오류를 확인한 후 월별 집계를 진행한다."
completed_order_sales = order_sales.loc[order_sales["order_status"].eq("completed")].copy()
assert len(completed_order_sales) > 0
completed_order_sales["order_month"] = completed_order_sales["order_date"].dt.to_period("M").astype(str)

completed_sales_items = completed_order_sales.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id", how="left", validate="many_to_one", indicator=True
)
products_merge_counts = completed_sales_items["_merge"].value_counts()
assert len(completed_sales_items) == len(completed_order_sales)
assert completed_sales_items["_merge"].eq("both").all()
assert completed_sales_items["order_item_id"].is_unique
completed_sales_items = completed_sales_items.drop(columns="_merge")
assert completed_sales_items[["category", "product_name", "order_month", "customer_id"]].notna().all().all()
assert completed_sales_items["customer_id"].isin(customers["customer_id"]).all()

In [5]:
# 주문·상품 병합의 행 수와 미매칭 확인
orders_probe = order_items.merge(
    orders, on="order_id", how="left", validate="many_to_one", indicator=True)
products_probe = completed_order_sales.merge(
    products, on="product_id", how="left", validate="many_to_one", indicator=True)

merge_check = pd.DataFrame([
    ["orders", orders_key_duplicates, len(order_items), len(orders_probe),
     int(orders_probe["_merge"].eq("both").sum()), int(orders_probe["_merge"].eq("left_only").sum())],
    ["products", products_key_duplicates, len(completed_order_sales), len(products_probe),
     int(products_probe["_merge"].eq("both").sum()), int(products_probe["_merge"].eq("left_only").sum())],
], columns=["병합 대상", "key 중복", "병합 전", "병합 후", "both", "미매칭"])
assert merge_check["병합 전"].eq(merge_check["병합 후"]).all()
assert merge_check["미매칭"].eq(0).all()
display(merge_check)
print("many_to_one + indicator 검증: PASS / 날짜 변환 실패:", date_failures)
print("completed:", completed_sales_items["order_id"].nunique(), "주문 /",
      len(completed_sales_items), "상세 행")

,병합 대상,key 중복,병합 전,병합 후,both,미매칭
0,orders,0,764,764,764,0
1,products,0,474,474,474,0


many_to_one + indicator 검증: PASS / 날짜 변환 실패: 0
completed: 184 주문 / 474 상세 행


![merge 검증](images/step03_merge.png)

### 결과 관찰

오른쪽 key 중복과 미매칭이 없고 두 병합 모두 상세 행 수가 유지되었다. 날짜 변환 실패도 0건이었다. completed 범위는 주문 184건, 상세 474행이다.

### 나의 해석과 판단

오른쪽 key가 고유하고 병합 전후 상세 ID와 행 수가 유지되어 행 증식이나 누락은 발견되지 않았다. 764행에서 474행으로 줄어든 것은 completed 필터의 결과이지 병합 손실이 아니다.

### 업무·분석적 의미

잘못된 병합은 동일한 금액을 중복 합산하거나 일부 거래를 빠뜨릴 수 있다. 집계 전에 관계·행 수·미매칭을 확인해야 한다.

### 한계와 추가 확인 사항

병합 성공은 연결 관계가 맞다는 근거이며 원본 값의 내용까지 정확하다는 뜻은 아니다.

## 4. completed 주문 범위와 집계

- 분석 범위 정의: 확보한 전체 기간의 completed 주문 184건, 상세 474행을 모든 집계에 동일하게 사용했다.
- 카테고리별 결과: 스포츠가 total_sales 31,743,000, total_quantity 295개, 금액 비중 21.31%로 금액 1위다.
- 상품별 결과: 완료 주문이 있는 상품은 98개다. 스포츠 상품 041이 금액 5,705,000, 수량 35개로 금액 1위다.
- 월별 결과: 2025년 9월~2026년 9월의 13개 월을 집계했다. 2026년 4월이 금액 16,634,000으로 가장 높고 고유 주문은 20건이다.
- 고객별 결과: 완료 주문 고객은 100명이다. 가상 고객 ID 117이 금액 4,100,000, 고유 주문 5건으로 금액 1위다.

In [6]:
# completed 주문의 카테고리별 금액과 수량 집계
category_sales = (
    completed_sales_items.groupby("category", as_index=False, dropna=False)
    .agg(total_sales=("line_total", "sum"), total_quantity=("quantity", "sum"))
    .sort_values(["total_sales", "category"], ascending=[False, True]).reset_index(drop=True)
)
category_sales["sales_share_pct"] = (
    category_sales["total_sales"] / completed_sales_items["line_total"].sum() * 100
).round(2)
display(category_sales)
print("금액 1위:", category_sales.iloc[0]["category"])

,category,total_sales,total_quantity,sales_share_pct
0,스포츠,31743000,295,21.31
1,전자기기,26400000,259,17.72
2,생활용품,23915000,272,16.05
3,뷰티,23383000,223,15.69
4,식품,16573000,133,11.12
5,도서,16389000,149,11.00
6,패션,10587000,111,7.11


금액 1위: 스포츠


In [7]:
product_sales = (
    completed_sales_items.groupby(["product_id", "product_name", "category"], as_index=False, dropna=False)
    .agg(total_sales=("line_total", "sum"), total_quantity=("quantity", "sum"))
    .sort_values(["total_sales", "product_id"], ascending=[False, True]).reset_index(drop=True)
)
print("completed 주문이 있는 상품:", len(product_sales), "개")
display(product_sales)  # 98개 상품 전체 결과

completed 주문이 있는 상품: 98 개


,product_id,product_name,category,total_sales,total_quantity
0,41,스포츠 상품 041,스포츠,5705000,35
1,12,식품 상품 012,식품,4375000,25
2,9,스포츠 상품 009,스포츠,3860000,20
3,72,뷰티 상품 072,뷰티,3780000,20
4,71,전자기기 상품 071,전자기기,3703000,23
5,68,스포츠 상품 068,스포츠,3640000,26
6,81,전자기기 상품 081,전자기기,3630000,22
7,11,패션 상품 011,패션,3565000,31
8,22,생활용품 상품 022,생활용품,3248000,29
9,89,생활용품 상품 089,생활용품,3090000,30


In [8]:
monthly_summary = (
    completed_sales_items.groupby("order_month", as_index=False, dropna=False)
    .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
    .sort_values("order_month").reset_index(drop=True)
)
display(monthly_summary)
print("전체 주문 데이터 기간:", orders["order_date"].min(), "~", orders["order_date"].max())

,order_month,total_sales,order_count
0,2025-09,3529000,5
1,2025-10,16197000,16
2,2025-11,11694000,17
3,2025-12,12319000,14
4,2026-01,5144000,8
5,2026-02,10034000,12
6,2026-03,14254000,18
7,2026-04,16634000,20
8,2026-05,14367000,19
9,2026-06,8943000,9


전체 주문 데이터 기간: 2025-09-15 ~ 2026-09-15


In [9]:
customer_sales = (
    completed_sales_items.groupby("customer_id", as_index=False, dropna=False)
    .agg(total_sales=("line_total", "sum"), order_count=("order_id", "nunique"))
    .sort_values(["total_sales", "customer_id"], ascending=[False, True]).reset_index(drop=True)
)
print("completed 주문 고객:", len(customer_sales), "명 / 이름 등 개인정보 미사용")
display(customer_sales)  # 가상 고객 ID별 100명 전체 결과

completed 주문 고객: 100 명 / 이름 등 개인정보 미사용


,customer_id,total_sales,order_count
0,117,4100000,5
1,102,3996000,4
2,83,3880000,4
3,30,3590000,5
4,40,3523000,4
5,20,3191000,2
6,3,3178000,2
7,111,3153000,3
8,66,3093000,4
9,147,2990000,2


![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰

카테고리별 금액은 스포츠 31,743,000, 전자기기 26,400,000, 생활용품 23,915,000 순으로 높았다. 패션은 10,587,000으로 가장 낮았다. 스포츠는 수량도 295개로 가장 많았다.

월별 금액은 2026년 4월에 가장 높았다. 고객별 집계에서는 구매 금액과 고유 주문 수를 함께 확인했다.

### 나의 해석과 판단

핵심 질문에 대한 답은 스포츠다. 다만 금액은 수량과 단가가 함께 반영된 결과이므로 금액 순위를 선호도나 수익성 순위로 확대해서 해석하지 않았다. 한 주문에 여러 상세 행이 있어 주문 건수는 nunique로 계산했다.

### 업무·분석적 의미

카테고리별 금액 구성을 파악하고 기여도가 큰 상품의 판매 내역·재고를 추가 점검할 수 있다. 월별 금액과 주문 수, 고객별 금액과 구매 빈도는 후속 분석의 기초 자료로 활용할 수 있다.

### 한계와 추가 확인 사항

할인·세금·배송비·원가가 없어 total_sales를 회계상 순매출이나 이익으로 단정할 수 없다. 상품 노출·재고 차이도 반영하지 못한다. 완료 주문이 없는 2개 상품은 상품별 집계에 포함되지 않는다.

전체 주문 데이터 기간은 2025-09-15~2026-09-15다. 양 끝의 9월은 불완전한 월이므로 다른 달과 단순 비교해 계절성이나 성장 추세를 판단하지 않았다.

## 5. 총합 일치 검증

- 원본 completed `line_total` 합계: 148,990,000.
- 카테고리 합계: 148,990,000.
- 월별 합계: 148,990,000.
- 고객별 합계: 148,990,000.
- 차이 여부: 상품별 합계 148,990,000을 포함한 5개 합계가 모두 일치하며 차이는 0이다.

원본 주문 상세에서 completed 주문 ID에 해당하는 행을 직접 선택하여 기준 합계를 계산했다.

In [10]:
# 원본 completed 금액과 집계 결과의 총합 비교
completed_ids = orders.loc[orders["order_status"].eq("completed"), "order_id"]
original_completed = raw_items.loc[raw_items["order_id"].isin(completed_ids)].copy()
original_total = (original_completed["quantity"] * original_completed["unit_price"]).sum()
totals = pd.DataFrame({
    "구분": ["원본 completed", "카테고리별", "상품별", "월별", "고객별"],
    "total_sales": [original_total, category_sales["total_sales"].sum(),
                    product_sales["total_sales"].sum(), monthly_summary["total_sales"].sum(),
                    customer_sales["total_sales"].sum()]
})
totals["원본과 차이"] = totals["total_sales"] - original_total
totals["일치"] = totals["원본과 차이"].eq(0)
assert totals["일치"].all()
assert set(original_completed["order_item_id"]) == set(completed_sales_items["order_item_id"])
assert set(completed_ids) == set(completed_sales_items["order_id"])
assert monthly_summary["order_count"].sum() == len(completed_ids)
assert customer_sales["order_count"].sum() == len(completed_ids)
display(totals)
print("PASS: 5개 합계 일치 / 원본 상세 ID 유지 / 고유 주문 수:", len(completed_ids))

,구분,total_sales,원본과 차이,일치
0,원본 completed,148990000,0,True
1,카테고리별,148990000,0,True
2,상품별,148990000,0,True
3,월별,148990000,0,True
4,고객별,148990000,0,True


PASS: 5개 합계 일치 / 원본 상세 ID 유지 / 고유 주문 수: 184


![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단

같은 주문 범위를 사용했다면 집계 기준을 바꾸어도 전체 금액이 같아야 한다. 병합 전 원본과 비교하고 상세 ID까지 확인하여 금액 누락·중복 가능성을 점검했다. 월별·고객별 주문 건수 합계도 각각 184건이었다.

불일치가 발생하면 주문 상태 필터 → key 중복·미매칭 → 날짜·그룹 key 결측 → 집계식 순서로 확인하겠다. 총합 일치만으로 원본 값이나 카테고리 분류 자체의 정확성까지 보장되지는 않는다.

실습 가이드에 따라 4개 집계표를 `reports/`에 CSV로 저장하고 다시 읽어 결과가 유지되는지도 확인했다.

In [11]:
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)
report_tables = {
    "ch04_category_sales.csv": category_sales,
    "ch04_product_sales.csv": product_sales,
    "ch04_monthly_sales.csv": monthly_summary,
    "ch04_customer_sales.csv": customer_sales,
}
saved = []
for filename, table in report_tables.items():
    path = REPORT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    reread = pd.read_csv(path)
    pd.testing.assert_frame_equal(table.reset_index(drop=True), reread, check_dtype=False)
    assert reread["total_sales"].sum() == original_total
    saved.append({"파일": filename, "행": len(reread), "저장·재읽기": "PASS"})
display(pd.DataFrame(saved))

,파일,행,저장·재읽기
0,ch04_category_sales.csv,7,PASS
1,ch04_product_sales.csv,98,PASS
2,ch04_monthly_sales.csv,13,PASS
3,ch04_customer_sales.csv,100,PASS


## 6. LLM pandas 코드 검증

- LLM Prompt 요약: 실제 컬럼으로 line_total을 계산하고, many-to-one 병합·completed 필터·고유 주문 수·원본 총합 검증을 포함한 집계 코드를 요청했다.
- 제안 코드 요약: Codex가 작성한 본 Notebook의 병합·필터·groupby 코드를 사용했다.
- 실제 컬럼/범위와 맞지 않은 부분: 실습 Notebook의 검토 예시 `order_items.groupby("category")`는 원본에 category가 없어 그대로 사용할 수 없다. 주문 상태도 orders에서 연결해야 한다.
- 수정한 내용: 원본에 바로 groupby하는 예시 대신 products를 연결하고 completed만 선택하는 흐름을 적용했다. key 중복·미매칭·행 수·날짜와 총합 검증을 포함했다.
- 최종 판단: 사용. 아래 검증 항목을 통과한 코드만 분석에 사용했다.

In [12]:
# LLM 코드의 컬럼·분석 범위·집계 기준 검증
review = pd.DataFrame([
    ["실제 컬럼 사용", all(set(cols).issubset(datasets[n].columns) for n, cols in required.items())],
    ["many-to-one 및 key 고유성", orders_key_duplicates == products_key_duplicates == 0],
    ["병합 행 수·미매칭", bool(merge_check["병합 전"].eq(merge_check["병합 후"]).all() and merge_check["미매칭"].eq(0).all())],
    ["날짜 변환 성공", date_failures == 0],
    ["completed만 포함", bool(completed_sales_items["order_status"].eq("completed").all())],
    ["월·고객 주문 수 nunique", bool(monthly_summary["order_count"].sum() == customer_sales["order_count"].sum() == len(completed_ids))],
    ["5개 총합 일치", bool(totals["일치"].all())],
], columns=["검증 항목", "통과"])
assert review["통과"].all()
display(review)
print(f"필터 생략(잘못된 범위): {order_items['line_total'].sum():,}")
print(f"completed만(분석 범위): {original_total:,}")
print("PASS: 실행 성공과 별개로 컬럼·범위·관계·집계 단위를 확인함")

,검증 항목,통과
0,실제 컬럼 사용,True
1,many-to-one 및 key 고유성,True
2,병합 행 수·미매칭,True
3,날짜 변환 성공,True
4,completed만 포함,True
5,월·고객 주문 수 nunique,True
6,5개 총합 일치,True


필터 생략(잘못된 범위): 255,610,000
completed만(분석 범위): 148,990,000
PASS: 실행 성공과 별개로 컬럼·범위·관계·집계 단위를 확인함


![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단

7개 검증 항목이 통과했다. 필터를 생략한 금액은 255,610,000으로 completed 금액보다 106,620,000 컸다. 코드가 오류 없이 실행되어도 질문과 다른 범위를 계산할 수 있으므로 실제 컬럼·필터·병합 관계·집계 단위를 확인해야 한다. 자동 검증은 정한 조건만 확인하며 원본 자료의 사실 여부까지 보장하지 않는다.

## 7. Chapter 04 최종 인사이트

### 가장 의미 있다고 생각한 결과 2가지

1. 스포츠가 completed 주문 기준 금액 1위였다. 금액은 31,743,000, 전체 비중은 21.31%, 수량은 295개였다. 다만 수익성이나 인기 순위로 단정하지 않았다.
2. 분석 범위를 끝까지 유지하는 것이 중요했다. 원본과 4개 집계의 합계는 모두 148,990,000이지만 completed 필터를 생략하면 255,610,000이 된다. 같은 계산식도 포함하는 주문 상태에 따라 결과가 달라진다.

### 그 결과를 뒷받침하는 수치/표

4절 카테고리별 집계표, 5절 총합 일치 검증표, 6절 필터 적용 전후 금액 비교를 근거로 삼았다.

### 추가로 확인하고 싶은 질문

카테고리별 금액 차이는 주문 수, 주문당 수량, 판매 단가 중 어느 항목과 더 관련이 있는가?

### 현재 결과의 한계

가상 데이터의 일부 기간에 대한 결과이며 실제 시장을 대표하지 않는다. 할인·세금·배송비·원가를 별도로 반영하지 못해 순매출·이익·고객 선호도나 장기 추세를 단정할 수 없다.

## 최종 제출 체크

- [x] 핵심 셀 Output이 남아 있습니다.
- [x] merge와 총합 검증 Evidence가 있습니다.
- [x] 결과 관찰과 해석이 구분되어 있습니다.
- [x] LLM 코드를 검증했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [ ] 최종 Notebook 파일 URL을 제출합니다.